# Advanced EDA: Product Performance Benchmarking (YoY & Historical Mean)

## Executive Summary & Analytical Domain
This notebook conducts detailed **Product Performance Benchmarking** across annual operating cycles. Evaluating individual product sales against both historical performance baselines and prior-year actuals enables merchandisers to:
- Identify catalog growth drivers versus fading older product lines.
- Benchmark individual annual product revenue against all-time average annual sales.
- Quantify Year-over-Year (YoY) dollar change and directional momentum using analytic window functions (`LAG()`).

## Key Performance Indicators (KPIs) & Methodology
- **Current Sales (`current_sales`)**: Total revenue generated by a product in a given calendar year.
- **Historical Average Sales (`avg_sales`)**: Mean annual sales for that specific product across all active years (`AVG(current_sales) OVER (PARTITION BY product_name)`).
- **Difference from Historical Mean (`diff_avg`)**: `current_sales - avg_sales`.
- **Average Performance Flag (`avg_change`)**: Categorical classification: `'Above Avg'`, `'Below Avg'`, or `'Avg'`.
- **Prior Year Sales (`previous_sales`)**: Revenue generated in the immediately preceding operating year (`LAG(current_sales) OVER (PARTITION BY product_name ORDER BY order_year)`).
- **YoY Dollar Change (`diff_py`)**: `current_sales - previous_sales`.
- **YoY Growth Flag (`py_change`)**: Categorical indicator: `'Increase'`, `'Decrease'`, or `'No Change'`.

## Data Contracts & Layer Dependencies
- **Target Layer**: `gold`
- **Tables Referenced**:
  - `gold.fact_sales` (`order_date`, `sales_amount`, `product_key`)
  - `gold.dim_products` (`product_key`, `product_name`)

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Setup display helper and session verification
try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure gold database context
spark.sql("USE gold")
print("Session configured. Active database context: gold")

## 1. Product Performance Benchmarking: Target Mean & YoY Analysis

### Business Context & Objective
Computes annual sales per product and benchmarks each annual result against both its historical mean and its previous year's sales performance using window partitions.

In [ ]:
# COMMAND ----------
# 1. Product Performance Benchmarking Query
df_product_performance = spark.sql("""
WITH yearly_product_sales AS (
    SELECT
        extract(year from f.order_date) AS order_year,
        p.product_name,
        sum(f.sales_amount) AS current_sales
    FROM gold.fact_sales f
    LEFT JOIN gold.dim_products p
        ON f.product_key = p.product_key
    WHERE f.order_date IS NOT NULL
    GROUP BY extract(year from f.order_date), p.product_name
)
SELECT
    order_year,
    product_name,
    current_sales,
    avg(current_sales) OVER (
        PARTITION BY product_name
    ) AS avg_sales,
    current_sales - avg(current_sales) OVER (
        PARTITION BY product_name
    ) AS diff_avg,
    CASE
        WHEN current_sales - avg(current_sales) OVER (PARTITION BY product_name) > 0 THEN 'Above Avg'
        WHEN current_sales - avg(current_sales) OVER (PARTITION BY product_name) < 0 THEN 'Below Avg'
        ELSE 'Avg'
    END AS avg_change,
    LAG(current_sales) OVER (
        PARTITION BY product_name
        ORDER BY order_year
    ) AS previous_sales,
    current_sales - LAG(current_sales) OVER (
        PARTITION BY product_name
        ORDER BY order_year
    ) AS diff_py,
    CASE
        WHEN current_sales - LAG(current_sales) OVER (PARTITION BY product_name ORDER BY order_year) > 0 THEN 'Increase'
        WHEN current_sales - LAG(current_sales) OVER (PARTITION BY product_name ORDER BY order_year) < 0 THEN 'Decrease'
        ELSE 'No Change'
    END AS py_change
FROM yearly_product_sales
ORDER BY
    product_name,
    order_year
""")
display(df_product_performance)

### Strategic Insights & Interpretation
- **Above-Average Performers**: Products that exceed their historical average indicate market share expansion and rising consumer demand.
- **Lag Analysis**: The first year of data for each product returns `NULL` for `previous_sales` resulting in `'No Change'`, accurately reflecting the lack of a baseline prior year.
- **Declining Products**: Consecutive years with `'Decrease'` flags highlight candidates for promotional clearance, redesign, or phase-out.

## 2. Standout Growth Products (Top YoY Gainers)

### Business Context & Objective
Isolates product-year observations with the largest positive YoY sales expansion (`diff_py > 0`) to identify top revenue accelerators.

In [ ]:
# COMMAND ----------
# 2. Standout Growth Accelerators (Top 10 YoY Gainers)
df_growth_leaders = spark.sql("""
WITH yearly_product_sales AS (
    SELECT
        extract(year from f.order_date) AS order_year,
        p.product_name,
        sum(f.sales_amount) AS current_sales
    FROM gold.fact_sales f
    LEFT JOIN gold.dim_products p
        ON f.product_key = p.product_key
    WHERE f.order_date IS NOT NULL
    GROUP BY extract(year from f.order_date), p.product_name
),
benchmarked AS (
    SELECT
        order_year,
        product_name,
        current_sales,
        LAG(current_sales) OVER (
            PARTITION BY product_name
            ORDER BY order_year
        ) AS previous_sales,
        current_sales - LAG(current_sales) OVER (
            PARTITION BY product_name
            ORDER BY order_year
        ) AS diff_py
    FROM yearly_product_sales
)
SELECT
    order_year,
    product_name,
    current_sales,
    previous_sales,
    diff_py
FROM benchmarked
WHERE diff_py > 0
ORDER BY diff_py DESC
LIMIT 10
""")
display(df_growth_leaders)

### Business Recommendations & Summary
1. **Catalog Rationalization**: Conduct regular reviews of products displaying persistent negative `diff_py` and `diff_avg` metrics.
2. **Inventory Stocking Priority**: Prioritize safety stock allocation for top YoY gainers to prevent costly stockouts.
3. **Marketing Budget Reinvestment**: Reinvest marketing capital into products that demonstrate proven positive customer adoption curves.